# Import the data from CIQ from the excel sheet

In [2]:
import pandas as pd
import os

cwd = os.getcwd()
cwd

'c:\\Users\\adamf\\PycharmProjects\\FedContracts_AlgoTrading'

In [2]:
filepath = rf"{cwd}\datsets\fundamentals\ciq_eps_revision_raw.csv"

In [3]:
raw = pd.read_csv(filepath, encoding="cp1252")
raw.head()

,SPGTable,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,Entity Name,Entity ID,Exchange,SP_SECTOR,Market Capitalization ($M),CIQ EPS Est - # of Est (actual),EPS_FY+1_Current,EPS_FY+1_1M_ago,EPS_FY+1_3M_ago,EPS_MEDIAN_EST,EPS_HIGH_EST,EPS_LOW_EST,EPS_SURPRISE,AS_OF
1,SP_ENTITY_NAME,SP_ENTITY_ID,SP_EXCHANGE,NaN,SP_MARKETCAP,SP_EPS_NUM_EST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,Current,FY+1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,Health Care,"2,658.52",15,-0.97,-0.49,-0.52,-0.98,-0.69,-1.22,28.57,06/03/2026


In [4]:
raw.columns = raw.iloc[0,:]
raw.drop([0,1,2,3], axis=0, inplace=True)
raw.head()

,Entity Name,Entity ID,Exchange,SP_SECTOR,Market Capitalization ($M),CIQ EPS Est - # of Est (actual),EPS_FY+1_Current,EPS_FY+1_1M_ago,EPS_FY+1_3M_ago,EPS_MEDIAN_EST,EPS_HIGH_EST,EPS_LOW_EST,EPS_SURPRISE,AS_OF
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,Health Care,"2,658.52",15,-0.97,-0.49,-0.52,-0.98,-0.69,-1.22,28.57,06/03/2026
5,1st Source Corporation (NASDAQGS:SRCE),100444,NASDAQGS,Financials,"1,640.83",3,6.64,6.64,6.48,6.65,6.73,6.55,4.01,06/03/2026
6,3M Company (NYSE:MMM),105135,NYSE,Industrials,"80,801.05",18,8.66,8.67,8.04,8.64,8.99,8.55,0.50,06/03/2026
7,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,Industrials,"9,816.41",14,4.02,4.03,3.80,4.01,4.13,3.90,1.32,06/03/2026
8,"A10 Networks, Inc. (NYSE:ATEN)",4433033,NYSE,Information Technology,"1,468.21",6,1.01,1.01,0.88,1.02,1.02,1.00,2.27,06/03/2026


In [ ]:
# Clean column names
raw.columns = [c.strip().lower().replace(" ", "_").replace("+", "") for c in raw.columns]

# Rename to consistent names
raw = raw.rename(columns={
    "entity_name": "entity_name_raw",
    "entity_id": "iqid",
    "market_capitalization_($m)": "mktcap_usd_m",
    "ciq_eps_est_-_#_of_est_(actual)": "n_est_fy1",
    "as_of": "asof_date"
})


# Ensure numeric
num_cols = ['mktcap_usd_m', 'n_est_fy1', 
    'eps_fy1_current', 'eps_fy1_1m_ago', 'eps_fy1_3m_ago', 
    'eps_median_est', 'eps_high_est', 'eps_low_est', 'eps_surprise'
]


# remove thousands separators, then convert safely
raw[num_cols] = (
    raw[num_cols]
    .replace({',': ''}, regex=True)
    .apply(pd.to_numeric, errors='coerce')
    .astype(float)
)

# Drop obvious junk rows
raw = raw.dropna() #(subset=["iqid"])

# Optional: remove cases where denominator was ~0, which can explode revisions
raw = raw[raw["eps_fy1_1m_ago"].abs() > 1e-6]
raw = raw[raw["eps_fy1_3m_ago"].abs() > 1e-6]

raw.head()

(1881, 14)
(1838, 14)


,entity_name_raw,iqid,exchange,sp_sector,mktcap_usd_m,n_est_fy1,eps_fy1_current,eps_fy1_1m_ago,eps_fy1_3m_ago,eps_median_est,eps_high_est,eps_low_est,eps_surprise,asof_date
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,Health Care,2658.52,15.0,-0.97,-0.49,-0.52,-0.98,-0.69,-1.22,28.57,06/03/2026
5,1st Source Corporation (NASDAQGS:SRCE),100444,NASDAQGS,Financials,1640.83,3.0,6.64,6.64,6.48,6.65,6.73,6.55,4.01,06/03/2026
6,3M Company (NYSE:MMM),105135,NYSE,Industrials,80801.05,18.0,8.66,8.67,8.04,8.64,8.99,8.55,0.50,06/03/2026
7,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,Industrials,9816.41,14.0,4.02,4.03,3.80,4.01,4.13,3.90,1.32,06/03/2026
8,"A10 Networks, Inc. (NYSE:ATEN)",4433033,NYSE,Information Technology,1468.21,6.0,1.01,1.01,0.88,1.02,1.02,1.00,2.27,06/03/2026


In [9]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1837 entries, 4 to 1884
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   entity_name_raw  1837 non-null   object 
 1   iqid             1837 non-null   object 
 2   exchange         1837 non-null   object 
 3   sp_sector        1837 non-null   object 
 4   mktcap_usd_m     1837 non-null   float64
 5   n_est_fy1        1837 non-null   float64
 6   eps_fy1_current  1837 non-null   float64
 7   eps_fy1_1m_ago   1837 non-null   float64
 8   eps_fy1_3m_ago   1837 non-null   float64
 9   eps_median_est   1837 non-null   float64
 10  eps_high_est     1837 non-null   float64
 11  eps_low_est      1837 non-null   float64
 12  eps_surprise     1837 non-null   float64
 13  asof_date        1837 non-null   object 
dtypes: float64(9), object(5)
memory usage: 215.3+ KB


# Store this in a database

In [1]:
import duckdb

In [17]:
con = duckdb.connect("research.duckdb")

con.execute("DROP TABLE IF EXISTS eps_raw")

con.execute("""
CREATE TABLE eps_raw AS
SELECT * FROM raw
""")

con.close()

In [18]:
con = duckdb.connect("research.duckdb")

result = con.execute(""" 
SELECT
    entity_name_raw,
    eps_fy1_current,
FROM eps_raw
LIMIT 10
""").df()

print(result)

                            entity_name_raw  eps_fy1_current
0         10x Genomics, Inc. (NASDAQGS:TXG)            -0.97
1    1st Source Corporation (NASDAQGS:SRCE)             6.64
2                     3M Company (NYSE:MMM)             8.66
3        A. O. Smith Corporation (NYSE:AOS)             4.02
4            A10 Networks, Inc. (NYSE:ATEN)             1.01
5                AAON, Inc. (NASDAQGS:AAON)             1.99
6                      AAR Corp. (NYSE:AIR)             4.71
7            Abbott Laboratories (NYSE:ABT)             5.68
8                   AbbVie Inc. (NYSE:ABBV)            14.53
9  AbCellera Biologics Inc. (NASDAQGS:ABCL)            -0.70


In [35]:
con.execute("""
            
CREATE OR REPLACE TABLE eps_features AS
WITH rev_table AS (
    SELECT
        *,
        (eps_fy1_current - eps_fy1_1m_ago) / ABS(eps_fy1_1m_ago) AS rev_1m,
        (eps_fy1_current - eps_fy1_3m_ago) / ABS(eps_fy1_3m_ago) AS rev_3m
    FROM eps_raw
)
SELECT
    *,
    (rev_1m - AVG(rev_1m) OVER (PARTITION BY sp_sector))
        / STDDEV_POP(rev_1m) OVER (PARTITION BY sp_sector) AS rev_1m_z_score,
    (rev_3m - AVG(rev_3m) OVER (PARTITION BY sp_sector))
        / STDDEV_POP(rev_3m) OVER (PARTITION BY sp_sector) AS rev_3m_z_score
            

FROM rev_table;
            
""").df()

,Count
0,1837


In [36]:
con.execute("""
            
SELECT *
FROM eps_features
LIMIT 10;
            
"""
).df()

,entity_name_raw,iqid,exchange,sp_sector,mktcap_usd_m,n_est_fy1,eps_fy1_current,eps_fy1_1m_ago,eps_fy1_3m_ago,eps_median_est,eps_high_est,eps_low_est,eps_surprise,asof_date,rev_1m,rev_3m,rev_1m_z_score,rev_3m_z_score
0,"Charter Communications, Inc. (NASDAQGS:CHTR)",4121481,NASDAQGS,Communication Services,29408.91,12.0,43.55,43.40,36.03,42.30,52.43,37.39,1.09,06/03/2026,0.003456,0.208715,-0.187050,-0.124376
1,Tencent Music Entertainment Group (NYSE:TME),9962087,NYSE,Communication Services,21096.08,22.0,0.86,0.86,0.86,0.86,0.91,0.78,3.59,06/03/2026,0.000000,0.000000,-0.188497,-0.135352
2,"Clear Channel Outdoor Holdings, Inc. (NYSE:CCO)",4121662,NYSE,Communication Services,1181.42,4.0,-0.09,0.04,0.04,-0.10,-0.05,-0.12,0.00,06/03/2026,-3.250000,-3.250000,-1.549232,-0.306262
3,Comcast Corporation (NASDAQGS:CMCSA),4057180,NASDAQGS,Communication Services,114807.27,17.0,3.68,3.68,4.22,3.66,3.94,3.51,2.62,06/03/2026,0.000000,-0.127962,-0.188497,-0.142081
4,"DoubleVerify Holdings, Inc. (NYSE:DV)",28172255,NYSE,Communication Services,1783.49,8.0,1.12,0.96,0.96,1.15,1.30,0.93,-8.33,06/03/2026,0.166667,0.166667,-0.118716,-0.126587
5,EchoStar Corporation (NASDAQGS:SATS),4190757,NASDAQGS,Communication Services,30690.17,2.0,0.41,-40.41,-39.24,0.41,2.66,-1.85,-24.75,06/03/2026,1.010146,1.010449,0.234438,-0.082215
6,Electronic Arts Inc. (NASDAQGS:EA),4100183,NASDAQGS,Communication Services,49550.24,21.0,8.59,8.63,8.61,8.55,9.83,7.72,7.76,06/03/2026,-0.004635,-0.002323,-0.190438,-0.135474
7,Formula One Group (NASDAQGS:FWON.K),4996388,NASDAQGS,Communication Services,20824.53,7.0,1.65,2.21,2.23,1.63,1.91,1.41,-2.25,06/03/2026,-0.253394,-0.260090,-0.294590,-0.149030
8,Fox Corporation (NASDAQGS:FOXA),13518181,NASDAQGS,Communication Services,24201.06,17.0,4.63,4.62,4.47,4.66,4.86,4.32,6.22,06/03/2026,0.002165,0.035794,-0.187591,-0.133470
9,Grindr Inc. (NYSE:GRND),5264620,NYSE,Communication Services,2149.56,5.0,0.55,0.49,0.49,0.54,0.60,0.49,-10.42,06/03/2026,0.122449,0.122449,-0.137230,-0.128913


# Creating universe for all dates

In [ ]:
filepath = rf"{cwd}\datsets\fundamentals\whole_universe.csv"

universe_df = pd.read_csv(filepath, encoding="cp1252")

con = duckdb.connect("research.duckdb")

con.execute("DROP TABLE IF EXISTS universe")

con.execute("""
CREATE TABLE universe AS
SELECT AS_OF, company_name, IQID FROM universe_df
""").df()


,Count
0,7497


In [15]:
con.execute("DROP TABLE IF EXISTS test")


In [17]:
con = duckdb.connect("research.duckdb")
con.execute(
    """
WITH
unique_dates AS (
    SELECT AS_OF FROM universe
),
unique_assets AS (
    SELECT company_name, IQID FROM universe
)

SELECT
    d.AS_OF,
    a.company_name,
    a.IQID
FROM unique_dates d
CROSS JOIN unique_assets a
ORDER BY a.company_name, d.AS_OF

"""
).df()

RuntimeError: Query interrupted